# Web Scraping Assignment: Springer Journal Data Collection

## Purpose

This project aims to collect and analyze data from the Springer Journal of Economics and Human Biology (https://link.springer.com/journal/12134/articles). The goal is to:

1. **Collect data** about published academic articles including:
   - Article titles
   - Authors
   - Paper types (e.g., Research Article, Review, etc.)
   - Publication dates

2. **Create a structured database** suitable for analysis

3. **Analyze the data** to gain insights about publishing trends, author contributions, and paper types

4. **Visualize findings** to communicate insights effectively

## Academic Context

Understanding publication patterns in academic journals can help researchers:
- Identify active research areas
- Track publication trends over time
- Understand the distribution of different types of academic work
- Analyze author collaboration patterns

## 1. Import Required Libraries

We'll use several Python libraries for this project:
- `requests`: To fetch web pages
- `BeautifulSoup`: To parse HTML content
- `pandas`: For data manipulation and storage
- `matplotlib` and `seaborn`: For data visualization

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import time
import re

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 2. Define Web Scraping Functions

We'll create functions to:
1. Fetch the web page content
2. Parse the HTML to extract article information
3. Handle multiple pages of articles

In [ ]:
def fetch_page(url):
    """
    Fetch the content of a web page.
    
    Parameters:
    url (str): The URL to fetch
    
    Returns:
    BeautifulSoup object or None if request fails
    """
    try:
        # Add headers to mimic a browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Parse the HTML content
        soup = BeautifulSoup(response.content, 'html.parser')
        return soup
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

In [ ]:
def extract_article_data(article_element):
    """
    Extract data from a single article element.
    
    Parameters:
    article_element: BeautifulSoup element containing article information
    
    Returns:
    dict: Dictionary containing article data
    """
    article_data = {
        'title': None,
        'authors': None,
        'paper_type': None,
        'publication_date': None
    }
    
    try:
        # Extract title
        title_elem = article_element.find('a', {'class': 'c-card__link'})
        if not title_elem:
            title_elem = article_element.find('h3', {'class': 'c-card__title'})
        if title_elem:
            article_data['title'] = title_elem.get_text(strip=True)
        
        # Extract authors
        authors_elem = article_element.find('ul', {'class': 'c-author-list'})
        if not authors_elem:
            authors_elem = article_element.find('span', {'class': 'c-author-list'})
        if authors_elem:
            authors = [author.get_text(strip=True) for author in authors_elem.find_all('li')]
            if not authors:  # Try alternative structure
                authors_text = authors_elem.get_text(strip=True)
                authors = [a.strip() for a in authors_text.split(',')]
            article_data['authors'] = ', '.join(authors) if authors else None
        
        # Extract paper type
        type_elem = article_element.find('span', {'class': 'c-meta__type'})
        if not type_elem:
            type_elem = article_element.find('span', string=re.compile('Article|Review|Editorial|Letter', re.I))
        if type_elem:
            article_data['paper_type'] = type_elem.get_text(strip=True)
        
        # Extract publication date
        date_elem = article_element.find('time')
        if not date_elem:
            date_elem = article_element.find('span', {'class': 'c-meta__item'})
        if date_elem:
            date_text = date_elem.get_text(strip=True)
            article_data['publication_date'] = date_text
    
    except Exception as e:
        print(f"Error extracting article data: {e}")
    
    return article_data

In [ ]:
def scrape_springer_journal(base_url, num_pages=3):
    """
    Scrape article data from Springer journal pages.
    
    Parameters:
    base_url (str): The base URL of the journal
    num_pages (int): Number of pages to scrape
    
    Returns:
    list: List of dictionaries containing article data
    """
    all_articles = []
    
    for page in range(1, num_pages + 1):
        # Construct URL for pagination
        if page == 1:
            url = base_url
        else:
            url = f"{base_url}?page={page}"
        
        print(f"Scraping page {page}: {url}")
        
        # Fetch the page
        soup = fetch_page(url)
        
        if soup is None:
            print(f"Failed to fetch page {page}")
            continue
        
        # Find all article elements
        articles = soup.find_all('article', {'class': 'c-card'})
        if not articles:
            articles = soup.find_all('div', {'class': 'app-article-list-row__item'})
        
        print(f"Found {len(articles)} articles on page {page}")
        
        # Extract data from each article
        for article in articles:
            article_data = extract_article_data(article)
            if article_data['title']:  # Only add if we got at least a title
                all_articles.append(article_data)
        
        # Be respectful to the server - add a delay between requests
        time.sleep(2)
    
    return all_articles

## 3. Collect Data from Springer Journal

Now we'll run the scraper to collect data from the journal. We'll scrape a few pages to get a representative sample of articles.

In [ ]:
# Define the journal URL
journal_url = "https://link.springer.com/journal/12134/articles"

# Scrape the data (adjust num_pages based on how much data you want)
print("Starting data collection...\n")
articles_data = scrape_springer_journal(journal_url, num_pages=3)

print(f"\nData collection complete! Collected {len(articles_data)} articles.")

## 4. Create a Structured Database

We'll convert the collected data into a pandas DataFrame, which serves as our database for analysis.

In [ ]:
# Create DataFrame from collected data
df = pd.DataFrame(articles_data)

# Display basic information about the dataset
print("Dataset Overview:")
print(f"Total number of articles: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:")
print(df.dtypes)

# Display first few rows
print("\nFirst 5 articles:")
df.head()

In [ ]:
# Check for missing values
print("Missing values in dataset:")
print(df.isnull().sum())
print(f"\nPercentage of missing values:")
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
# Save the raw data to CSV for future use
csv_filename = 'springer_journal_articles.csv'
df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"Data saved to {csv_filename}")

## 5. Data Cleaning and Preprocessing

Before analysis, we need to clean and preprocess the data.

In [ ]:
# Create a copy for processing
df_clean = df.copy()

# Remove rows with missing titles (essential field)
df_clean = df_clean.dropna(subset=['title'])

# Fill missing paper types with 'Unknown'
df_clean['paper_type'] = df_clean['paper_type'].fillna('Unknown')

# Process publication dates
def parse_date(date_str):
    """Try to extract year from various date formats"""
    if pd.isna(date_str):
        return None
    try:
        # Try to find a 4-digit year
        year_match = re.search(r'\b(19|20)\d{2}\b', str(date_str))
        if year_match:
            return int(year_match.group())
    except:
        pass
    return None

df_clean['year'] = df_clean['publication_date'].apply(parse_date)

# Count number of authors
def count_authors(author_str):
    if pd.isna(author_str):
        return 0
    return len([a.strip() for a in str(author_str).split(',') if a.strip()])

df_clean['num_authors'] = df_clean['authors'].apply(count_authors)

print(f"Cleaned dataset has {len(df_clean)} articles")
df_clean.head()

## 6. Data Analysis

Let's analyze the collected data to gain insights.

### 6.1 Summary Statistics

In [ ]:
# Basic statistics
print("=" * 50)
print("SUMMARY STATISTICS")
print("=" * 50)

print(f"\nTotal articles collected: {len(df_clean)}")
print(f"\nNumber of authors statistics:")
print(df_clean['num_authors'].describe())

if df_clean['year'].notna().any():
    print(f"\nPublication year range: {df_clean['year'].min():.0f} - {df_clean['year'].max():.0f}")

print(f"\nPaper types distribution:")
print(df_clean['paper_type'].value_counts())

### 6.2 Top Authors

In [ ]:
# Extract individual authors and count their publications
all_authors = []
for authors_str in df_clean['authors'].dropna():
    all_authors.extend([a.strip() for a in str(authors_str).split(',')])

author_counts = pd.Series(all_authors).value_counts().head(10)

print("Top 10 Most Prolific Authors:")
print(author_counts)

## 7. Data Visualization

Visual representations help us understand patterns in the data more easily.

### 7.1 Distribution of Paper Types

In [ ]:
# Create a pie chart for paper types
plt.figure(figsize=(10, 6))
paper_type_counts = df_clean['paper_type'].value_counts()

colors = sns.color_palette('Set2', len(paper_type_counts))
plt.pie(paper_type_counts.values, labels=paper_type_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
plt.title('Distribution of Paper Types', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

### 7.2 Number of Authors per Article

In [ ]:
# Histogram of number of authors
plt.figure(figsize=(10, 6))
plt.hist(df_clean['num_authors'], bins=range(0, df_clean['num_authors'].max() + 2), 
         edgecolor='black', color='skyblue', alpha=0.7)
plt.xlabel('Number of Authors', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Number of Authors per Article', fontsize=16, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Most common number of authors: {df_clean['num_authors'].mode()[0]}")
print(f"Average number of authors: {df_clean['num_authors'].mean():.2f}")

### 7.3 Publications Over Time (if year data available)

In [ ]:
# Only create this visualization if we have year data
if df_clean['year'].notna().sum() > 0:
    plt.figure(figsize=(12, 6))
    year_counts = df_clean['year'].value_counts().sort_index()
    
    plt.bar(year_counts.index, year_counts.values, color='coral', edgecolor='black', alpha=0.7)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Number of Publications', fontsize=12)
    plt.title('Publications Over Time', fontsize=16, fontweight='bold')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Year data not available for time series visualization")

### 7.4 Top Authors Bar Chart

In [ ]:
# Bar chart of top authors
if len(author_counts) > 0:
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(author_counts)), author_counts.values, color='lightgreen', edgecolor='black')
    plt.yticks(range(len(author_counts)), author_counts.index)
    plt.xlabel('Number of Publications', fontsize=12)
    plt.ylabel('Author', fontsize=12)
    plt.title('Top 10 Most Prolific Authors', fontsize=16, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Author data not available for visualization")

## 8. Key Findings and Conclusions

Based on the collected data and analysis:

In [ ]:
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

print(f"\n1. Dataset Size:")
print(f"   - Successfully collected data on {len(df_clean)} articles")

print(f"\n2. Paper Types:")
most_common_type = df_clean['paper_type'].value_counts().index[0]
print(f"   - Most common paper type: {most_common_type}")
print(f"   - Distribution: {df_clean['paper_type'].value_counts().to_dict()}")

print(f"\n3. Author Collaboration:")
print(f"   - Average authors per paper: {df_clean['num_authors'].mean():.2f}")
print(f"   - Most common number of authors: {df_clean['num_authors'].mode()[0]}")
print(f"   - Range: {df_clean['num_authors'].min()} to {df_clean['num_authors'].max()} authors")

if df_clean['year'].notna().sum() > 0:
    print(f"\n4. Publication Timeline:")
    print(f"   - Year range: {df_clean['year'].min():.0f} - {df_clean['year'].max():.0f}")
    if len(year_counts) > 0:
        most_productive_year = year_counts.idxmax()
        print(f"   - Most productive year: {most_productive_year:.0f} ({year_counts.max()} articles)")

print("\n" + "=" * 60)

## 9. Data Export and Database Storage

The collected data has been saved in multiple formats for different use cases.

In [ ]:
# Save cleaned data
df_clean.to_csv('springer_journal_articles_cleaned.csv', index=False, encoding='utf-8')
print("✓ Cleaned data saved to: springer_journal_articles_cleaned.csv")

# Save as Excel for easier viewing
try:
    df_clean.to_excel('springer_journal_articles.xlsx', index=False, engine='openpyxl')
    print("✓ Data saved to Excel: springer_journal_articles.xlsx")
except:
    print("  Excel export skipped (openpyxl not installed)")

# Save summary statistics
with open('analysis_summary.txt', 'w', encoding='utf-8') as f:
    f.write("SPRINGER JOURNAL DATA ANALYSIS SUMMARY\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Total articles collected: {len(df_clean)}\n")
    f.write(f"Data collection date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("Paper Types Distribution:\n")
    f.write(str(df_clean['paper_type'].value_counts()) + "\n\n")
    f.write("Author Statistics:\n")
    f.write(f"Average authors per paper: {df_clean['num_authors'].mean():.2f}\n")
    f.write(f"Most common number of authors: {df_clean['num_authors'].mode()[0]}\n")
    
print("✓ Summary saved to: analysis_summary.txt")

print("\n" + "=" * 60)
print("DATA COLLECTION AND ANALYSIS COMPLETE!")
print("=" * 60)

## 10. Conclusion

This project successfully demonstrated:

1. **Web Scraping Skills**: Collected structured data from a real academic journal website
2. **Data Processing**: Cleaned and organized raw data into a usable database
3. **Data Analysis**: Extracted meaningful insights about publication patterns
4. **Visualization**: Created clear visualizations to communicate findings

### Potential Extensions:
- Scrape more pages to get a larger dataset
- Analyze abstract text for common keywords or topics
- Build a network graph of author collaborations
- Compare trends across different time periods
- Perform sentiment analysis on article titles

### Technical Notes:
- The scraper respects the website by including delays between requests
- Error handling ensures the script continues even if some pages fail
- Data is saved in multiple formats for flexibility
- The code is modular and can be adapted for other Springer journals